<a href="https://colab.research.google.com/github/BSsangani/How-Data-Quality-Affects-Energy-Demand-Forecasting-in-UK-Smart-Grids/blob/main/step3_defect_injection_imputation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3: Injecting Data Defects and Testing Imputation

This is where the actual experiment starts. I'm taking the clean baseline
from Step 1 and making damaged copies of it, four different ways, then
trying to fix each one, four different ways, and checking how close the
fix gets back to the real values.

**Defects:** random missing (5%, 15%, 30%), a block gap (24-48 hours),
Gaussian noise, and a timestamp shift.

**Fixes:** mean fill, forward-fill, interpolation, Kalman filter.

Needs `output/clean_demand_2024_2025.csv` from Step 1 -- this notebook
pulls it straight from Google Drive, so it doesn't need to be re-uploaded
every session.

## 0. Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/energy-project')

# quick check that everything's where it should be
print("Current folder:", os.getcwd())
print("Files in output:", os.listdir('output'))

Mounted at /content/drive
Current folder: /content/drive/MyDrive/energy-project
Files in output: ['clean_demand_2024_2025.csv', 'best_hyperparameters.json']


## 1. Installs

In [2]:
!pip install -q pandas numpy statsmodels

## 2. Load the clean baseline

In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)  # same results every run

clean_data = pd.read_csv("output/clean_demand_2024_2025.csv", parse_dates=["datetime"])
nd_values = clean_data["ND"].values.copy().astype(float)

os.makedirs("output/degraded", exist_ok=True)
os.makedirs("output/imputed", exist_ok=True)

print(f"Loaded {len(nd_values):,} readings.")

Loaded 35,088 readings.


## 3. The defect functions

Four separate functions, one per defect type, so each one only does its own job.

In [4]:
def add_random_missing(values, percent):
    damaged = values.copy()
    n_missing = int(len(values) * percent / 100)
    positions = np.random.choice(len(values), size=n_missing, replace=False)
    damaged[positions] = np.nan
    return damaged


def add_block_gap(values, min_hours=24, max_hours=48):
    # one continuous stretch goes missing, like a sensor going offline
    damaged = values.copy()
    gap_periods = np.random.randint(min_hours * 2, max_hours * 2)  # 2 periods/hour
    start = np.random.randint(0, len(values) - gap_periods)
    damaged[start:start + gap_periods] = np.nan
    return damaged


def add_gaussian_noise(values, noise_std=500):
    noise = np.random.normal(0, noise_std, size=len(values))
    return values + noise


def add_timestamp_shift(values, max_shift_periods=4, percent_affected=10):
    # swapping values between nearby rows -- what a timestamp error looks
    # like once everything's back in time order
    damaged = values.copy()
    n_affected = int(len(values) * percent_affected / 100)
    positions = np.random.choice(len(values) - max_shift_periods, size=n_affected, replace=False)
    for pos in positions:
        shift = np.random.randint(-max_shift_periods, max_shift_periods + 1)
        swap_pos = pos + shift
        if 0 <= swap_pos < len(values):
            damaged[pos], damaged[swap_pos] = damaged[swap_pos], damaged[pos]
    return damaged

print("Defect functions ready.")

Defect functions ready.


## 4. Create all 6 damaged versions and save them

In [5]:
print("Creating damaged versions...\n")
damaged_versions = {}

for percent in [5, 15, 30]:
    name = f"missing_{percent}pct"
    damaged_versions[name] = add_random_missing(nd_values, percent)
    print(f"Created: {name}")

damaged_versions["block_gap"] = add_block_gap(nd_values)
print("Created: block_gap")

damaged_versions["gaussian_noise"] = add_gaussian_noise(nd_values)
print("Created: gaussian_noise")

damaged_versions["timestamp_shift"] = add_timestamp_shift(nd_values)
print("Created: timestamp_shift")

for name, damaged_nd in damaged_versions.items():
    out_df = clean_data.copy()
    out_df["ND"] = damaged_nd
    out_df.to_csv(f"output/degraded/{name}.csv", index=False)

print("\nSaved to output/degraded/")

Creating damaged versions...

Created: missing_5pct
Created: missing_15pct
Created: missing_30pct
Created: block_gap
Created: gaussian_noise
Created: timestamp_shift

Saved to output/degraded/


## 5. The imputation functions

Four ways to try to fix the gaps -- from the simplest (mean) to the smartest (Kalman filter).

In [6]:
def impute_mean(values):
    fixed = values.copy()
    mean_value = np.nanmean(values)
    fixed[np.isnan(fixed)] = mean_value
    return fixed


def impute_forward_fill(values):
    series = pd.Series(values)
    fixed = series.ffill()
    fixed = fixed.fillna(np.nanmean(values))  # in case gaps are right at the start
    return fixed.values


def impute_interpolation(values):
    series = pd.Series(values)
    fixed = series.interpolate(method="linear")
    fixed = fixed.fillna(np.nanmean(values))
    return fixed.values


def impute_kalman(values):
    # predicts from the underlying trend rather than just copying/averaging --
    # the "smart" option. statsmodels' state-space model handles NaNs natively.
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    series = pd.Series(values)
    model = SARIMAX(series, order=(1, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False)
    smoothed = fitted.get_prediction().predicted_mean
    fixed = series.copy()
    fixed[fixed.isna()] = smoothed[fixed.isna()]
    fixed = fixed.fillna(np.nanmean(values))
    return fixed.values


imputation_methods = {
    "mean_fill": impute_mean,
    "forward_fill": impute_forward_fill,
    "interpolation": impute_interpolation,
    "kalman_filter": impute_kalman,
}

print("Imputation functions ready.")

Imputation functions ready.


## 6. Run every imputation method on every defect, and measure recovery

Noise and timestamp-shift don't create actual gaps (no NaNs), so there's nothing for
these four methods to fill in -- those two defects skip straight to modelling instead
in the next step.

In [7]:
print("Running imputation...\n")
recovery_results = []

for defect_name, damaged_nd in damaged_versions.items():

    was_missing = np.isnan(damaged_nd)
    if was_missing.sum() == 0:
        print(f"{defect_name}: no gaps, skipping (goes straight to modelling)")
        continue

    for method_name, method_function in imputation_methods.items():
        fixed_nd = method_function(damaged_nd)

        true_values = nd_values[was_missing]
        recovered_values = fixed_nd[was_missing]
        rmse = float(np.sqrt(np.mean((true_values - recovered_values) ** 2)))

        recovery_results.append({
            "defect": defect_name,
            "imputation_method": method_name,
            "recovery_rmse": round(rmse, 2),
        })

        out_df = clean_data.copy()
        out_df["ND"] = fixed_nd
        out_df.to_csv(f"output/imputed/{defect_name}__{method_name}.csv", index=False)

        print(f"{defect_name:15s} + {method_name:15s} -> RMSE = {rmse:.2f}")

Running imputation...

missing_5pct    + mean_fill       -> RMSE = 5994.60
missing_5pct    + forward_fill    -> RMSE = 954.42
missing_5pct    + interpolation   -> RMSE = 256.61
missing_5pct    + kalman_filter   -> RMSE = 954.28
missing_15pct   + mean_fill       -> RMSE = 6083.03
missing_15pct   + forward_fill    -> RMSE = 1093.23
missing_15pct   + interpolation   -> RMSE = 308.71
missing_15pct   + kalman_filter   -> RMSE = 1092.33
missing_30pct   + mean_fill       -> RMSE = 6060.13
missing_30pct   + forward_fill    -> RMSE = 1380.50
missing_30pct   + interpolation   -> RMSE = 398.58
missing_30pct   + kalman_filter   -> RMSE = 1379.02
block_gap       + mean_fill       -> RMSE = 5944.92
block_gap       + forward_fill    -> RMSE = 6155.61
block_gap       + interpolation   -> RMSE = 5497.73
block_gap       + kalman_filter   -> RMSE = 6524.39
gaussian_noise: no gaps, skipping (goes straight to modelling)
timestamp_shift: no gaps, skipping (goes straight to modelling)


## 7. Save a summary and see which method wins for each defect

In [8]:
results_df = pd.DataFrame(recovery_results)
results_df = results_df.sort_values(["defect", "recovery_rmse"])
results_df.to_csv("output/imputation_recovery_summary.csv", index=False)

print("Saved output/imputation_recovery_summary.csv\n")
print("Best method per defect:")
best_per_defect = results_df.loc[results_df.groupby("defect")["recovery_rmse"].idxmin()]
print(best_per_defect.to_string(index=False))

Saved output/imputation_recovery_summary.csv

Best method per defect:
       defect imputation_method  recovery_rmse
    block_gap     interpolation        5497.73
missing_15pct     interpolation         308.71
missing_30pct     interpolation         398.58
 missing_5pct     interpolation         256.61


## 8. Everything's already saved to Drive

Since this notebook is working directly out of the Drive-mounted folder, there's
nothing left to manually download and re-upload -- `output/degraded/`,
`output/imputed/`, and `output/imputation_recovery_summary.csv` are already sitting
in `energy-project/output` in Drive, ready for Step 4 (running all 4 models on
every degraded/imputed version).